### 准备数据集

In [2]:
# 配置代码
import os

os.environ['http_proxy'] = '127.0.0.1:10809'
os.environ['https_proxy'] = '127.0.0.1:10809'

In [3]:
# 加载tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('hfl/rbt3')

In [4]:
tokenizer

BertTokenizerFast(name_or_path='hfl/rbt3', vocab_size=21128, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'})

In [5]:
# 试用
tokenizer.batch_encode_plus(['明月装饰了你的窗子', '你装饰了别人的梦'], 
                           truncation=True)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'input_ids': [[101, 3209, 3299, 6163, 7652, 749, 872, 4638, 4970, 2094, 102], [101, 872, 6163, 7652, 749, 1166, 782, 4638, 3457, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [6]:
# 从磁盘加载中文数据集
from datasets import load_from_disk

dataset = load_from_disk('./data/ChnSentiCorp/')

In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9600
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 0
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1200
    })
})

In [8]:
# 缩小数据规模, 便于测试. 
dataset['train'] = dataset['train'].shuffle().select(range(2000))
dataset['test'] = dataset['test'].shuffle().select(range(100))

In [9]:
dataset['train'][0]

{'text': '书的名字绝对讨好人，我相信很多人和我一样，因为书名而买了这本书。书里的内容没有什么新意，是你我小时候爸妈，老师教导的那套。可以当立志的读本。',
 'label': 0}

In [12]:
def f(data, tokenizer):
    return tokenizer.batch_encode_plus(data['text'], truncation=True)

dataset = dataset.map(f, batched=True,
                     batch_size=1000,
                     num_proc=4,
                     remove_columns=['text'],
                     fn_kwargs={'tokenizer': tokenizer})

dataset

Map (num_proc=4):   0%|          | 0/2000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['label'],
        num_rows: 0
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 100
    })
})

In [13]:
# 删掉太长的句子
def f(data):
    return [len(i) <= 512 for i in data['input_ids']]

dataset = dataset.filter(f, batched=True, batch_size=1000, num_proc=4)

Filter (num_proc=4):   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

In [14]:
from transformers import AutoModelForSequenceClassification

In [15]:
model = AutoModelForSequenceClassification.from_pretrained('hfl/rbt3', num_labels=2)

Some weights of the model checkpoint at hfl/rbt3 were not used when initializing BertForSequenceClassification: ['cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/rbt3

In [16]:
# 统计模型参数量
sum([i.nelement() for i in model.parameters()])

38478338

In [18]:
# 模型试算
import torch

# 模拟一条数据
data = {
    'input_ids': torch.ones(4, 10, dtype=torch.long),
    'token_type_ids': torch.ones(4, 10, dtype=torch.long),
    'attention_mask': torch.ones(4, 10, dtype=torch.long),
    'labels': torch.ones(4, dtype=torch.long)
}

out = model(**data)

In [19]:
out

SequenceClassifierOutput(loss=tensor(0.8028, grad_fn=<NllLossBackward0>), logits=tensor([[0.2630, 0.0546],
        [0.2630, 0.0546],
        [0.2630, 0.0546],
        [0.2630, 0.0546]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [20]:
out['loss']

tensor(0.8028, grad_fn=<NllLossBackward0>)

In [23]:
out['logits']

tensor([[0.2630, 0.0546],
        [0.2630, 0.0546],
        [0.2630, 0.0546],
        [0.2630, 0.0546]], grad_fn=<AddmmBackward0>)

In [24]:
# 定义评价函数
from datasets import load_metric

metric = load_metric('accuracy')

D:\.venv\lib\site-packages\ipykernel_launcher.py:4: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  after removing the cwd from sys.path.


In [34]:
import numpy as np
from transformers.trainer_utils import EvalPrediction

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = logits.argmax(axis=1)
    # 不用metric, 自己算也行
    return {'accuracy': (pred == labels).mean()}
#     print(pred)
#     print(labels)
#     return metric.compute(predctions=pred, references=labels)


# 模拟输出

eval_pred = EvalPrediction(
    predictions=np.array([[0, 1], [2, 3], [4, 5], [6, 7]]),
    label_ids=np.array([1, 1, 0, 1])
)
compute_metrics(eval_pred)

{'accuracy': 0.75}

In [33]:
# 这个版本的准确率有bug. 暂时不用. 
predictions=np.array([0, 0, 1, 0])
references = np.array([1, 0, 1, 0])
metric.compute(predctions=predictions, references=references)

TypeError: 'NoneType' object is not iterable

In [35]:
# 定义训练参数
from transformers import TrainingArguments

In [37]:
args = TrainingArguments(
    # 定义临时数据保存路径
    output_dir = './output_dir',
    
    # 定义测试执行策略, 可取 no, epoch, steps
    evaluation_strategy = 'steps', 
    
    # 定义每隔多少个step执行一次测试
    eval_steps = 30,
    
    # 定义模型保存策略, 可取 no, epoch, steps
    save_strategy = 'steps',
    
    # 定义每隔多少个step保存一次
    save_steps = 30,
    
    # 定义总共训练集个轮次
    num_train_epochs = 1,
    
    # 定义学习率
    learning_rate = 1e-4,
    
    # 加入参数权重衰减, 防止过拟合
    weight_decay = 1e-2,
    
    # 定义训练和测试时候的批次大小
    per_device_eval_batch_size = 16,
    per_device_train_batch_size = 16,
    
    # 定义是否使用GPU训练
    no_cuda = False
)

In [38]:
# 定义训练器
from transformers import Trainer
from transformers.data.data_collator import DataCollatorWithPadding

In [40]:
trainer = Trainer(
    model = model,
    args = args,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    compute_metrics = compute_metrics,
    data_collator = DataCollatorWithPadding(tokenizer)
)

In [41]:
# 测试数据整理函数
data_collator = DataCollatorWithPadding(tokenizer)

In [43]:
# 获取一批数据
data = dataset['train'][:5]
print(data)

{'label': [0, 0, 1, 0, 1], 'input_ids': [[101, 741, 4638, 1399, 2099, 5318, 2190, 6374, 1962, 782, 8024, 2769, 4685, 928, 2523, 1914, 782, 1469, 2769, 671, 3416, 8024, 1728, 711, 741, 1399, 5445, 743, 749, 6821, 3315, 741, 511, 741, 7027, 4638, 1079, 2159, 3766, 3300, 784, 720, 3173, 2692, 8024, 3221, 872, 2769, 2207, 3198, 952, 4268, 1968, 8024, 5439, 2360, 3136, 2193, 4638, 6929, 1947, 511, 1377, 809, 2496, 4989, 2562, 4638, 6438, 3315, 511, 102], [101, 3141, 4178, 6206, 1962, 754, 10972, 6821, 763, 3322, 1690, 8024, 6134, 7481, 2523, 1117, 511, 852, 3221, 6858, 6814, 12311, 9683, 8165, 3466, 3844, 8476, 3946, 2428, 671, 5663, 1762, 8347, 118, 8419, 722, 7313, 8024, 4801, 4669, 1469, 712, 3352, 4638, 3946, 2428, 6772, 3633, 2382, 8024, 852, 3221, 8476, 2218, 2607, 2587, 749, 511, 2828, 9201, 1305, 3123, 1762, 6438, 1305, 1690, 7027, 7481, 2233, 4197, 1357, 679, 1139, 3341, 749, 8024, 3209, 1921, 6206, 2145, 3302, 3018, 2137, 511, 102], [101, 5050, 3221, 3867, 6897, 6438, 4289, 704, 6

In [44]:
# 查看句子长度
for i in data['input_ids']:
    print(len(i))

72
85
167
44
83


In [45]:
# 调用数据整理函数, 进行整理
data = data_collator(data)

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [46]:
# 查看整理之后的数据
for k, v in data.items():
    print(k, v.shape)

input_ids torch.Size([5, 167])
token_type_ids torch.Size([5, 167])
attention_mask torch.Size([5, 167])
labels torch.Size([5])


In [47]:
data

{'input_ids': tensor([[  101,   741,  4638,  1399,  2099,  5318,  2190,  6374,  1962,   782,
          8024,  2769,  4685,   928,  2523,  1914,   782,  1469,  2769,   671,
          3416,  8024,  1728,   711,   741,  1399,  5445,   743,   749,  6821,
          3315,   741,   511,   741,  7027,  4638,  1079,  2159,  3766,  3300,
           784,   720,  3173,  2692,  8024,  3221,   872,  2769,  2207,  3198,
           952,  4268,  1968,  8024,  5439,  2360,  3136,  2193,  4638,  6929,
          1947,   511,  1377,   809,  2496,  4989,  2562,  4638,  6438,  3315,
           511,   102,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,  

In [48]:
tokenizer.decode(data['input_ids'][0])

'[CLS] 书 的 名 字 绝 对 讨 好 人 ， 我 相 信 很 多 人 和 我 一 样 ， 因 为 书 名 而 买 了 这 本 书 。 书 里 的 内 容 没 有 什 么 新 意 ， 是 你 我 小 时 候 爸 妈 ， 老 师 教 导 的 那 套 。 可 以 当 立 志 的 读 本 。 [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]'

In [49]:
# 训练和测试

# 测试一把
trainer.evaluate()

***** Running Evaluation *****
  Num examples = 98
  Batch size = 16


{'eval_loss': 0.746782660484314,
 'eval_accuracy': 0.40816326530612246,
 'eval_runtime': 3.0309,
 'eval_samples_per_second': 32.334,
 'eval_steps_per_second': 2.31}

In [50]:
# 训练
trainer.train()

D:\.venv\lib\site-packages\transformers\optimization.py:310: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  FutureWarning,
***** Running training *****
  Num examples = 1978
  Num Epochs = 1
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 124
  Number of trainable parameters = 38478338


Step,Training Loss,Validation Loss,Accuracy
30,No log,0.487499,0.775510
60,No log,0.340572,0.867347
90,No log,0.305656,0.867347
120,No log,0.287684,0.846939


***** Running Evaluation *****
  Num examples = 98
  Batch size = 16
Saving model checkpoint to ./output_dir\checkpoint-30
Configuration saved in ./output_dir\checkpoint-30\config.json
Model weights saved in ./output_dir\checkpoint-30\pytorch_model.bin
***** Running Evaluation *****
  Num examples = 98
  Batch size = 16
Saving model checkpoint to ./output_dir\checkpoint-60
Configuration saved in ./output_dir\checkpoint-60\config.json
Model weights saved in ./output_dir\checkpoint-60\pytorch_model.bin
***** Running Evaluation *****
  Num examples = 98
  Batch size = 16
Saving model checkpoint to ./output_dir\checkpoint-90
Configuration saved in ./output_dir\checkpoint-90\config.json
Model weights saved in ./output_dir\checkpoint-90\pytorch_model.bin
***** Running Evaluation *****
  Num examples = 98
  Batch size = 16
Saving model checkpoint to ./output_dir\checkpoint-120
Configuration saved in ./output_dir\checkpoint-120\config.json
Model weights saved in ./output_dir\checkpoint-120\pyt

TrainOutput(global_step=124, training_loss=0.40998326578447897, metrics={'train_runtime': 32.7148, 'train_samples_per_second': 60.462, 'train_steps_per_second': 3.79, 'total_flos': 71826088671480.0, 'train_loss': 0.40998326578447897, 'epoch': 1.0})

In [51]:
trainer.evaluate()

***** Running Evaluation *****
  Num examples = 98
  Batch size = 16


{'eval_loss': 0.2862688899040222,
 'eval_accuracy': 0.8571428571428571,
 'eval_runtime': 0.4737,
 'eval_samples_per_second': 206.868,
 'eval_steps_per_second': 14.776,
 'epoch': 1.0}

In [52]:
# 保存和加载模型
import torch

In [53]:
trainer.save_model(output_dir='./output_dir/save_model')

Saving model checkpoint to ./output_dir/save_model
Configuration saved in ./output_dir/save_model\config.json
Model weights saved in ./output_dir/save_model\pytorch_model.bin


In [54]:
# 加载模型

model.load_state_dict(torch.load('./output_dir/save_model/pytorch_model.bin'))

<All keys matched successfully>

In [55]:
# 恢复中断训练
trainer.train(resume_from_checkpoint='./output_dir/checkpoint-90')

Loading model from ./output_dir/checkpoint-90.
***** Running training *****
  Num examples = 1978
  Num Epochs = 1
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 124
  Number of trainable parameters = 38478338
  Continuing training from checkpoint, will skip to saved global_step
  Continuing training from epoch 0
  Continuing training from global step 90
  Will skip the first 0 epochs then the first 90 batches in the first epoch. If this takes a lot of time, you can add the `--ignore_data_skip` flag to your launch command, but you will resume the training on data already seen by your model.


  0%|          | 0/90 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Accuracy
120,No log,0.277997,0.857143


***** Running Evaluation *****
  Num examples = 98
  Batch size = 16
Saving model checkpoint to ./output_dir\checkpoint-120
Configuration saved in ./output_dir\checkpoint-120\config.json
Model weights saved in ./output_dir\checkpoint-120\pytorch_model.bin


Training completed. Do not forget to share your model on huggingface.co/models =)




TrainOutput(global_step=124, training_loss=0.06748032569885254, metrics={'train_runtime': 8.8334, 'train_samples_per_second': 223.923, 'train_steps_per_second': 14.038, 'total_flos': 69834802852152.0, 'train_loss': 0.06748032569885254, 'epoch': 1.0})

In [56]:
trainer.evaluate()

***** Running Evaluation *****
  Num examples = 98
  Batch size = 16


{'eval_loss': 0.2799054682254791,
 'eval_accuracy': 0.8571428571428571,
 'eval_runtime': 0.5186,
 'eval_samples_per_second': 188.966,
 'eval_steps_per_second': 13.498,
 'epoch': 1.0}

In [57]:
# 预测
model.eval()

# 取一批数据做预测
for i, data in enumerate(trainer.get_eval_dataloader()):
    break
    
for k, v in data.items():
    # 因为model在gpu上训练的, 所以数据也要拷贝到gpu上
    data[k] = v.to('cuda')
    
# 预测
out = model(**data)
pred = out['logits'].argmax(dim=1)

In [58]:
pred

tensor([1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0], device='cuda:0')

In [61]:
for i in range(16):
    print(tokenizer.decode(data['input_ids'][i], skip_special_tokens=True))
    print('label=', data['labels'][i].item())
    print('predict=', pred[i].item())

这 本 书 中 的 图 片 很 让 人 触 动 ， 比 如 为 劳 拉 祈 祷 的 校 友 ， 惠 特 尼 和 马 特 的 笑 容 ， 很 动 人 。
label= 1
predict= 1
看 了 这 本 书 ， 很 有 感 触 ， 再 次 从 书 中 看 到 一 个 个 或 让 人 快 乐 、 或 让 人 感 动 、 或 让 人 感 慨 万 千 、 无 奈 长 叹 的 故 事 。 很 多 时 候 ， 成 人 总 是 以 为 自 己 的 迫 切 愿 望 也 是 孩 子 的 需 要 ， 我 们 总 是 将 自 己 的 观 点 、 自 己 的 好 恶 强 加 给 孩 子 ， 我 们 忘 了 ， 在 孩 子 的 眼 中 ， 这 个 世 界 与 成 人 并 不 一 样 。 我 想 ， 阅 读 这 本 书 ， 更 多 的 是 改 变 或 形 成 一 个 观 念 。 要 贴 近 孩 子 ， 要 了 解 孩 子 ， 做 孩 子 的 知 心 人 ， 好 朋 友 ， 用 与 孩 子 一 样 的 眼 睛 与 心 灵 来 体 验 与 观 察 这 个 世 界 。
label= 1
predict= 1
外 观 ， 配 置 ， 价 格 ， 三 个 组 合 起 来 看 是 绝 对 超 值 的 东 东 我 4699 入 手 ， 抢 到 了
label= 1
predict= 1
新 机 拿 到 手 就 有 硬 件 问 题 ， 而 且 等 了 6 天 才 到 货 ， 第 二 天 就 返 修 ， 到 现 在 还 没 得 到 处 理 意 见 ！
label= 0
predict= 0
机 器 的 发 热 量 小 ， 比 意 想 中 的 小 很 多 。 噪 音 控 制 的 很 好 ， 运 行 的 时 候 基 本 听 不 到 什 么 声 音 。 双 核 的 速 度 不 错 。 官 网 上 的 xp 驱 动 也 比 较 好 装 。
label= 1
predict= 1
初 五 住 的 ， 这 么 冷 的 天 没 有 暖 气 ， 没 有 热 水 。 携 程 预 定 前 台 说 没 有 收 到 。 总 之 感 觉 很 业 余 。
label= 0
predict= 0
怀 着 十 分 激 动 的 心 情 放 映 ， 可 是 看 着 看 着 发 现 ， 在 放 映 完 毕 后 ， 出 现 一 集 米 老 鼠 